In [0]:
import os
import requests
import time
from datetime import datetime
from bs4 import BeautifulSoup

# 1. Setup camera pages
CAMERAS = {
    "west_entrance": "https://www.nps.gov/media/webcam/view.htm?id=33478DF3-1DD8-B71B-0B8C97DB0A03B0F7",
    "logan_pass": "https://www.nps.gov/media/webcam/view.htm?id=325AE6AF-BAEB-F65D-EF3D638BF683E78E",
    "apgar_village": "https://www.nps.gov/media/webcam/view.htm?id=81B4692D-1DD8-B71B-0B9AE4B7C186B022"
}

# 2. Local workspace scratch path
STORAGE_DIR = "/tmp/glacier_webcam_images/"

def run_scraper():
    os.makedirs(STORAGE_DIR, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    print(f"--- Starting Ingestion Cycle at {timestamp} ---")
    
    for cam_name, page_url in CAMERAS.items():
        try:
            response = requests.get(page_url, timeout=15)
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')
                all_images = soup.find_all('img')
                
                full_image_url = None
                for img in all_images:
                    src = img.get('src', '')
                    if "webcam" in src.lower() or "jpg" in src.lower():
                        if src.startswith('/'):
                            full_image_url = "https://www.nps.gov" + src
                        else:
                            full_image_url = src
                        break
                
                if full_image_url:
                    img_response = requests.get(full_image_url, timeout=15)
                    if img_response.status_code == 200:
                        filename = f"{STORAGE_DIR}{cam_name}_{timestamp}.jpg"
                        with open(filename, "wb") as f:
                            f.write(img_response.content)
                        print(f"Successfully archived: {cam_name}")
                    else:
                        print(f"Image download failed for {cam_name}")
                else:
                    print(f"Link extraction failed for {cam_name}")
            else:
                print(f"Page load failed for {cam_name}")
                
        except Exception as e:
            print(f"Error on {cam_name}: {e}")

# 3. Main Loop
while True:
    run_scraper()
    print("Sleeping for 60 seconds until next webcam refresh...\n")
    time.sleep(60)

In [0]:
import os
print(os.listdir("/tmp/glacier_webcam_images/"))